# SChain v0.3.0 — persistent EXPAND with incremental Neo4j Save

This notebook keeps the live SQLite database on Colab's local disk and publishes verified SQLite-backup generations plus essential artifacts to Google Drive. Neo4j receives only validated accepted facts, one document at a time.

For the first execution choose `NEW_RUN` and a unique `EXPERIMENT_ID`. Later runtimes must reuse the same ID with `RESUME` (the default). Never put credentials in the generated YAML or Drive directory.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, shutil, subprocess, sys, time, urllib.request

drive.mount('/content/drive')
EXPERIMENT_ID = 'schain-expand-2026-10-19'  # keep stable across runtimes
RUN_MODE = 'RESUME'  # change to NEW_RUN only for a genuinely new experiment
BRANCH = 'mirza'
REPOSITORY = 'https://github.com/ErfanBabakhani/Schain.git'
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
MAX_MODEL_LEN = 16384  # lower to 8192 if the selected GPU cannot serve 16k
DRIVE_RUNS = Path('/content/drive/MyDrive/SChain/expand_runs')
LOCAL_RUN = Path('/content/schain_runs') / EXPERIMENT_ID
REPO = Path('/content/Schain')
assert RUN_MODE in {'NEW_RUN', 'RESUME'}
assert EXPERIMENT_ID and '/' not in EXPERIMENT_ID
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
print({'experiment_id': EXPERIMENT_ID, 'mode': RUN_MODE, 'drive': str(DRIVE_RUNS)})

In [ ]:
# Clone/fetch mirza. RESUME pins the exact commit recorded by NEW_RUN so a
# later branch update cannot silently change prompts or experiment behavior.
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', f'origin/{BRANCH}'], check=True)
metadata_path = DRIVE_RUNS / EXPERIMENT_ID / 'experiment.json'
if RUN_MODE == 'RESUME' and metadata_path.exists():
    required_commit = json.loads(metadata_path.read_text()).get('git_commit')
    if required_commit:
        subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', required_commit], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
status = subprocess.check_output(['git', '-C', str(REPO), 'status', '--short'], text=True).strip()
assert not status, f'Repository is not clean: {status}'
print('Checked out', BRANCH, commit)

In [ ]:
# Install SChain and vLLM. If Colab requests a runtime restart after vLLM installation,
# restart and rerun from the first cell with the same EXPERIMENT_ID and RESUME.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
try:
    import vllm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)
print('SChain and vLLM dependencies are installed')

## Neo4j credentials

Create Colab Secrets named `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD`, and grant this notebook access. They are placed only in process environment variables and are never written into Drive snapshots.

In [ ]:
from google.colab import userdata
for name, default in [('NEO4J_USERNAME', 'neo4j')]:
    try: os.environ[name] = userdata.get(name)
    except Exception: os.environ[name] = default
for name in ('NEO4J_URI', 'NEO4J_PASSWORD'):
    os.environ[name] = userdata.get(name)
assert os.environ.get('NEO4J_URI') and os.environ.get('NEO4J_PASSWORD')
os.environ['SCHAIN_AUDIT_TRACE'] = '1'
os.environ['SCHAIN_AUDIT_FULL_CONTENT'] = '1'
print('Neo4j secrets loaded (values intentionally hidden)')

In [ ]:
# Generate an untracked runtime overlay. Working files stay local; snapshots go to Drive.
import yaml
CONFIG = LOCAL_RUN / 'expand.runtime.yaml'
storage = {
    'raw_documents': str(LOCAL_RUN / 'raw/documents.jsonl'),
    'normalized_documents': str(LOCAL_RUN / 'processed/documents.parquet'),
    'extracted_facts': str(LOCAL_RUN / 'processed/facts.parquet'),
    'extractions': str(LOCAL_RUN / 'processed/extractions'),
    'search_results': str(LOCAL_RUN / 'processed/search_results.parquet'),
    'entity_resolution': str(LOCAL_RUN / 'processed/entity_resolution.parquet'),
    'rejected_facts': str(LOCAL_RUN / 'processed/rejected_facts.parquet'),
    'edge_support': str(LOCAL_RUN / 'processed/edge_support.parquet'),
    'verification_evidence': str(LOCAL_RUN / 'processed/verification_evidence.parquet'),
    'extraction_integrity': str(LOCAL_RUN / 'processed/extraction_integrity.json'),
    'state_db': str(LOCAL_RUN / 'state/state.db'),
    'run_config': str(LOCAL_RUN / 'state/run_config.json'),
    'graph_export': str(LOCAL_RUN / 'outputs/supply_chain_graph.v1.json'),
}
overlay = {
    'include': [str(REPO / 'config/colab_professor_expand.yaml')],
    'runtime_overlay': True,
    'save': {'enabled': True, 'backend': 'neo4j', 'mode': 'incremental',
             'max_attempts': 3, 'retry_backoff_seconds': 2.0},
    'checkpoint_store': {'enabled': True, 'experiment_id': EXPERIMENT_ID,
        'directory': str(DRIVE_RUNS), 'snapshot_every_documents': 1,
        'keep_snapshots': 3, 'lock_stale_seconds': 21600},
    'audit': {'enabled': True, 'capture_content': True,
              'path': str(LOCAL_RUN / 'audit')},
    'storage': storage,
}
CONFIG.write_text(yaml.safe_dump(overlay, sort_keys=False), encoding='utf-8')
print('Runtime config:', CONFIG)
print(CONFIG.read_text())

In [ ]:
# Verify Neo4j before starting an expensive expansion.
from neo4j import GraphDatabase
probe = GraphDatabase.driver(os.environ['NEO4J_URI'], auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD']))
probe.verify_connectivity()
probe.close()
print('Neo4j connectivity verified')

In [ ]:
# Start the OpenAI-compatible Qwen/vLLM endpoint if it is not already healthy.
def healthy():
    try:
        return urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2).status == 200
    except Exception:
        return False

VLLM_LOG = LOCAL_RUN / 'vllm.log'
if not healthy():
    log_handle = VLLM_LOG.open('a', encoding='utf-8')
    vllm_process = subprocess.Popen([
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL, '--served-model-name', MODEL, '--port', '8000',
        '--max-model-len', str(MAX_MODEL_LEN), '--gpu-memory-utilization', '0.90',
    ], stdout=log_handle, stderr=subprocess.STDOUT)
    for _ in range(120):
        if healthy(): break
        if vllm_process.poll() is not None:
            raise RuntimeError(f'vLLM exited; inspect {VLLM_LOG}')
        time.sleep(5)
assert healthy(), f'vLLM did not become healthy; inspect {VLLM_LOG}'
print('vLLM is serving', MODEL, 'with max model length', MAX_MODEL_LEN)

In [ ]:
# Run the real EXPAND CLI. The CLI restores before opening SQLite and snapshots
# after each document plus final graph/audit export.
cmd = [sys.executable, '-m', 'schain', 'expand', '--config', str(CONFIG), '--save',
       '--experiment-id', EXPERIMENT_ID, '--checkpoint-store', str(DRIVE_RUNS)]
cmd.append('--new-run' if RUN_MODE == 'NEW_RUN' else '--resume')
print('Executing:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=REPO, text=True)
if result.returncode:
    raise RuntimeError(f'SChain exited with status {result.returncode}')

In [ ]:
# Inspect durable resume and graph-save progress without exposing credentials.
import sqlite3, pandas as pd
db = Path(storage['state_db'])
with sqlite3.connect(db) as conn:
    frontier = dict(conn.execute('SELECT status, COUNT(*) FROM frontier GROUP BY status'))
    tiers = dict(conn.execute('SELECT tier, COUNT(*) FROM frontier GROUP BY tier ORDER BY tier'))
    documents = dict(conn.execute('SELECT status, COUNT(*) FROM documents GROUP BY status'))
    commits = dict(conn.execute('SELECT status, COUNT(*) FROM neo4j_commits GROUP BY status'))
facts_path = Path(storage['extracted_facts'])
accepted = len(pd.read_parquet(facts_path)) if facts_path.exists() else 0
latest_path = DRIVE_RUNS / EXPERIMENT_ID / 'LATEST.json'
latest = json.loads(latest_path.read_text()) if latest_path.exists() else {}
print(json.dumps({'experiment_id': EXPERIMENT_ID, 'frontier': frontier, 'tiers': tiers,
                  'documents': documents, 'accepted_facts': accepted,
                  'neo4j_commits': commits, 'durable_snapshot': latest}, indent=2))

In [ ]:
# Package Graph JSON, Audit JSON, metrics, and runtime config for download.
from google.colab import files
archive_base = Path('/content') / f'{EXPERIMENT_ID}-outputs'
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=LOCAL_RUN)
print('Final semantic graph:', storage['graph_export'])
print('Durable checkpoint:', DRIVE_RUNS / EXPERIMENT_ID)
files.download(archive)